# Week 7 — Advanced Models

Train XGBoost, try a few parameter settings, and compare its R² with Linear
Regression, Decision Tree, and Random Forest. This notebook works with the
original Notebooks 02–04; no edits to those notebooks are required.

Use the original features for this experiment because they gave the strongest
Random Forest result in Week 6. This is a starting choice, not proof that these
are the best features for XGBoost.

Save this beside `02_preprocessing.ipynb`, `03_baseline_model.ipynb`, and
`04_model_comparison.ipynb`. Notebook 01 must already have created
`data/processed/residential_single_family.csv`. If needed, run `%pip install xgboost`
once in a separate cell, restart the kernel, then run this notebook from the top.

## 1. Import libraries

Keep the pandas and plotting tools from the earlier weeks and add XGBoost.
`capture_output()` hides routine printouts and tables from the notebooks we run.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
from IPython.display import display, Markdown
from IPython.utils.io import capture_output

## 2. Reuse the existing preprocessing notebook

Notebook 02 creates the original final training/test sets and also leaves
`clean_df` available before imputation and encoding. Use that earlier dataframe
for validation so validation values cannot influence the tuning preparation.
The final prepared CSVs are refreshed by Notebook 02 as usual.

`%run` creates its variables in this notebook's Python session. VS Code may
underline those imported names even though running the cell defines them.

In [ ]:
with capture_output():
    %run ./02_preprocessing.ipynb

print("Final training:", test_month - X, "through", test_month - 1)
print("Final test month:", test_month)

## 3. Set aside the validation month

AVM sections 01 and 09: select parameters using the month before the test month.
With the current data, tuning training is May 2025–February 2026, validation is
March 2026, and testing is April 2026. After tuning, the final model can train on
all 11 months before April. The monthly input exports must be complete.

Learn the tuning price cutoffs from the earlier training months only (AVM section 03).
Keep the original final X_train/X_test/y_train/y_test unchanged during tuning.

In [ ]:
assert X >= 2, "At least two months before the test month are needed."
validation_month = test_month - 1
months = clean_df["CloseDate"].dt.to_period("M")

tuning_data = clean_df.loc[
    (months >= test_month - X) & (months < validation_month)
].copy()
validation_data = clean_df.loc[months == validation_month].copy()
assert tuning_data["CloseDate"].dt.to_period("M").nunique() == X - 1

tune_lower, tune_upper = tuning_data["ClosePrice"].quantile([0.005, 0.995])
tuning_data = tuning_data.loc[
    tuning_data["ClosePrice"].between(tune_lower, tune_upper)
].copy()
validation_data = validation_data.loc[
    validation_data["ClosePrice"].between(tune_lower, tune_upper)
].copy()
assert len(tuning_data) > 1 and len(validation_data) > 1
print("Tuning training rows:", len(tuning_data))
print("Validation month:", validation_month, "Rows:", len(validation_data))

## 4. Prepare validation using the same rules

This is the only preparation repeated here. Slicing the already imputed X_train
would let March influence its own replacement values and category list. Relearn
medians, room-count modes, and categories from May–February instead.

AVM sections 06–07: flag missing values, fill them using training values, group
rare categories, and align one-hot columns. ClosePrice and CloseDate are excluded
from the predictors. Scaling is not needed for the tree model.

In [ ]:
tune_fill_values = tuning_data[numeric_columns].median()
for column in ["BedroomsTotal", "BathroomsTotalInteger"]:
    tune_fill_values[column] = tuning_data[column].mode().iloc[0]
assert tune_fill_values.notna().all()

for data in [tuning_data, validation_data]:
    for column in numeric_columns + categorical_columns:
        data[column + "_Missing"] = data[column].isna().astype(int)
    data[numeric_columns] = data[numeric_columns].fillna(tune_fill_values)
    data[categorical_columns] = data[categorical_columns].fillna("Unknown")

for column in categorical_columns:
    counts = tuning_data[column].value_counts()
    keep_categories = counts[counts >= minimum_count].index
    for data in [tuning_data, validation_data]:
        keep = data[column].isin(keep_categories) | data[column].eq("Unknown")
        data[column] = data[column].where(keep, "Other")

X_tune = pd.get_dummies(
    tuning_data.drop(columns=["ClosePrice", "CloseDate"]),
    columns=categorical_columns, dtype=bool
)
X_validation = pd.get_dummies(
    validation_data.drop(columns=["ClosePrice", "CloseDate"]),
    columns=categorical_columns, dtype=bool
).reindex(columns=X_tune.columns, fill_value=False)
y_tune = tuning_data["ClosePrice"]
y_validation = validation_data["ClosePrice"]

assert X_tune.columns.equals(X_validation.columns)
assert not X_tune.isna().any().any()
assert not X_validation.isna().any().any()

## 5. Try four settings and choose using validation R²

XGBoost adds trees sequentially to improve its predictions. Try moderate depths
and learning rates, then give the smaller learning rate more trees. These four
hand-selected trials are light tuning, not an exhaustive search or proven optima.

| Parameter | Meaning |
|---|---|
| max_depth | Maximum depth of each tree; deeper trees can learn more detail. |
| learning_rate | Size of each tree's contribution; smaller steps may need more trees. |
| n_estimators | Number of boosting trees. |

`hist` uses histogram-based tree building; `reg:squarederror` trains a price
regressor. Fix the seed for reproducibility and use four CPU threads. Other
parameters retain the defaults of the installed XGBoost version.

In [ ]:
parameter_choices = [
    {"max_depth": 4, "learning_rate": 0.10, "n_estimators": 200},
    {"max_depth": 6, "learning_rate": 0.10, "n_estimators": 200},
    {"max_depth": 6, "learning_rate": 0.05, "n_estimators": 200},
    {"max_depth": 6, "learning_rate": 0.05, "n_estimators": 400}
]

tuning_results = []
for parameters in parameter_choices:
    print("Trying:", parameters, flush=True)
    candidate = XGBRegressor(
        **parameters, objective="reg:squarederror", tree_method="hist",
        random_state=42, n_jobs=4
    )
    candidate.fit(X_tune, y_tune)
    tuning_results.append({
        **parameters,
        "Train_R2": candidate.score(X_tune, y_tune),
        "Validation_R2": r2_score(y_validation, candidate.predict(X_validation))
    })

tuning_results = pd.DataFrame(tuning_results)
display(tuning_results.round(4))
best_row = tuning_results.loc[tuning_results["Validation_R2"].idxmax()]
best_parameters = {
    "max_depth": int(best_row["max_depth"]),
    "learning_rate": float(best_row["learning_rate"]),
    "n_estimators": int(best_row["n_estimators"])
}
print("Selected parameters:", best_parameters)

## 6. Reuse the baseline and model-comparison notebooks

Now that settings are selected, run Notebooks 03–04 on the final CSVs from Notebook
02. Their model definitions and sanity checks are reused directly. This also
refreshes their result files. It may take several minutes on the full dataset.
The captured printouts and figures keep this notebook's output focused.

In [ ]:
with capture_output():
    %run ./03_baseline_model.ipynb
    %run ./04_model_comparison.ipynb

## 7. Train the selected XGBoost model and test it

Use the selected settings and the same final rows/features as the earlier models.
March is now part of final training. Evaluate April only after selecting settings;
do not change the settings in response to this test score (AVM section 09).

In [ ]:
xgb_model = XGBRegressor(
    **best_parameters, objective="reg:squarederror", tree_method="hist",
    random_state=42, n_jobs=4
)
xgb_model.fit(X_train, y_train)
xgb_predictions = xgb_model.predict(X_test)
xgb_train_r2 = xgb_model.score(X_train, y_train)
xgb_test_r2 = r2_score(y_test, xgb_predictions)

print(f"XGBoost training R²: {xgb_train_r2:.4f}")
print(f"XGBoost test R²: {xgb_test_r2:.4f}")

## 8. Compare the four models

Higher test R² is better. ChangeFromBaseline is an absolute R² difference.
TrainTestGap helps inspect possible overfitting or differences between months;
it does not prove why a model behaves differently on the test properties.

In [ ]:
advanced_comparison = pd.DataFrame({
    "Model": ["Linear Regression", "Decision Tree", "Random Forest", "XGBoost"],
    "Train_R2": [
        baseline_model.score(X_train, y_train), tree_train_r2,
        forest_train_r2, xgb_train_r2
    ],
    "Test_R2": [baseline_r2, tree_test_r2, forest_test_r2, xgb_test_r2]
})
advanced_comparison["ChangeFromBaseline"] = advanced_comparison["Test_R2"] - baseline_r2
advanced_comparison["TrainTestGap"] = (
    advanced_comparison["Train_R2"] - advanced_comparison["Test_R2"]
)
advanced_comparison["TrainingRows"] = len(X_train)
advanced_comparison["TestRows"] = len(X_test)
advanced_comparison["PredictorColumns"] = X_train.shape[1]
display(advanced_comparison.round(4))

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(
    advanced_comparison["Model"], advanced_comparison["Test_R2"],
    color=["gray", "darkorange", "seagreen", "steelblue"]
)
ax.bar_label(bars, fmt="%.4f", padding=4)
ax.set_title("Week 7: Test R² Comparison")
ax.set_ylabel("Test R²")
ax.set_ylim(min(0, advanced_comparison["Test_R2"].min() - 0.05), 1.05)
plt.tight_layout()
plt.show()

## 9. Save results and predictions

Save the tuning table, final comparison, and test predictions for Week 8. The
prior notebooks load y from CSV, so use arrays to keep dates and targets in the
same saved row order instead of aligning their different dataframe indexes.

In [ ]:
assert len(test_df) == len(y_test)
test_predictions = pd.DataFrame({
    "CloseDate": test_df["CloseDate"].to_numpy(),
    "ActualClosePrice": y_test.to_numpy(),
    "Linear Regression": baseline_model.predict(X_test),
    "Decision Tree": tree_predictions,
    "Random Forest": forest_predictions,
    "XGBoost": xgb_predictions
})
tuning_results.to_csv("../data/processed/xgboost_tuning_results.csv", index=False)
advanced_comparison.to_csv("../data/processed/advanced_model_comparison.csv", index=False)
test_predictions.to_csv("../data/processed/advanced_model_predictions.csv", index=False)

## 10. Summary

Generate a compact findings paragraph from this run's scores.

In [ ]:
display(Markdown(
    f"This week, I trained XGBoost and tested four combinations of depth, learning "
    f"rate, and tree count using **{validation_month}** for validation. The selected "
    f"settings were **{best_parameters}**. After retraining on all {X} preceding "
    f"months, XGBoost achieved a test R² of **{xgb_test_r2:.4f}** on **{test_month}**, "
    f"compared with **{baseline_r2:.4f}** for Linear Regression and "
    f"**{forest_test_r2:.4f}** for Random Forest. Its R² change was "
    f"**{xgb_test_r2 - baseline_r2:+.4f}** from the baseline and "
    f"**{xgb_test_r2 - forest_test_r2:+.4f}** from Random Forest. Its training/test "
    f"gap was **{xgb_train_r2 - xgb_test_r2:.4f}**. These results show whether this "
    f"experiment improved predictions on this month; improvement is not assumed."
))

Deeper trees and more rounds can learn more detailed patterns, but can also fit
details that do not generalize. A smaller learning rate takes smaller steps and
may need more rounds. Validation compares these tradeoffs before the final test.

This is a Week 7 experiment using the existing features and 11-month training
window. It does not optimize that window or establish statistical significance.
Prior weeks have already examined the same test month, so confirm conclusions
on later months and additional historical cutoffs. Scores cover the retained
price range after the frozen training cutoffs. Week 8 adds MAPE, MdAPE, and error
breakdowns. Keep the monthly data snapshot and project dependency versions fixed
when reproducing this experiment.

**References:** project timeline, Week 7; AVM best-practices sections 01, 03,
06–07, and 09–11; official XGBoost
[parameter documentation](https://xgboost.readthedocs.io/en/stable/parameter.html)
and [installation guide](https://xgboost.readthedocs.io/en/stable/install.html).

The file is numbered 06 because your existing Week 6 notebook is
`05_feature_engineering.ipynb`.